<a href="https://colab.research.google.com/github/hyunkoome/DL_Study/blob/dev/DL_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Tue Feb  4 01:25:38 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 565.90                 Driver Version: 565.90         CUDA Version: 12.7     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090      WDDM  |   00000000:01:00.0  On |                  Off |
|  0%   34C    P8             16W /  450W |     782MiB /  24564MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!nvcc -V

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Wed_Oct_30_01:18:48_Pacific_Daylight_Time_2024
Cuda compilation tools, release 12.6, V12.6.85
Build cuda_12.6.r12.6/compiler.35059454_0


In [ ]:
!pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

Looking in indexes: https://download.pytorch.org/whl/cu126


## **1. pytorch tensor 기본 연산**

In [ ]:
import torch

In [ ]:
x = torch.tensor([[1,0], [0, 1]], dtype=torch.float32)
y = torch.tensor([[5,6], [7, 8]], dtype=torch.float32)

# basic op
print(x+y)
print(x-y)
print(x * y) # multiple: element-wise
print(x @ y) # multiple

tensor([[6., 6.],
        [7., 9.]])
tensor([[-4., -6.],
        [-7., -7.]])
tensor([[5., 0.],
        [0., 8.]])
tensor([[5., 6.],
        [7., 8.]])


**ex01) 랜덤한 텐서 (3, 3)을 생성하고, 정규화를 수행하는 코드를 작성해 봐.**

In [ ]:
import torch
import numpy as np

# create random tensor
random_tensor = torch.rand(3, 3, dtype=torch.float32)
print(random_tensor)

# min-max normalization
min_val = random_tensor.min()
max_val = random_tensor.max()
normalized_tensor = (random_tensor-min_val)/(max_val-min_val)
print(normalized_tensor)

tensor([[0.7960, 0.9164, 0.5767],
        [0.0654, 0.5653, 0.2052],
        [0.4380, 0.4607, 0.7325]])
tensor([[0.8586, 1.0000, 0.6008],
        [0.0000, 0.5875, 0.1643],
        [0.4379, 0.4645, 0.7839]])


## **2. pytorch Autograd 자동 미분**

In [ ]:
# requres_grad=True 설정 시, 자동 미분

a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(3.0, requires_grad=True)

y = a**2 + b**3
y.backward()

print(a.grad) # dy/da
print(b.grad) # dy/db

tensor(4.)
tensor(27.)


**ex02) f(x) = x^3 + 2x^2 + 5x + 1 의 도함수를 자동 미분으로 구해보자.**

In [ ]:
import torch
import numpy as np

x = torch.tensor(2.0, requires_grad=True)

f_x = x**3 + 2*x**2 + 5*x + 1
f_x.backward()

print(f"f'(x) at x=2: {x.grad}") # d f_x /dx

f'(x) at x=2: 25.0


## **3. CNN을 이용한 이미지 분류**

MNIST 데이터셋을 이용하여 간단한 CNN(Convolutional Neural Network) 모델을 구현하고 학습시키는 코드를 작성

※ PyTorch 및 torchvision을 활용해야 함

**데이터 로드**
- torchvision.datasets.MNIST를 이용하여 MNIST 데이터셋을 다운로드
- 데이터 변환(transform)을 적용
  - torchvision.transforms를 사용하여 데이터를 Tensor로 변환해야 함
- torch.utils.data.DataLoader를 통해 배치 단위로 데이터를 로드
- 배치 크기는 32
- 데이터 로딩 시 shuffle=True로 설정

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms

# MNIST DataLoad
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)

**모델 구현**

- torch.nn.Module을 상속하여 SimpleCNN 클래스를 정의
- Conv2d
  - 입력 채널 1개
  - 출력 채널 16개
  - 3x3 커널
  - 패딩 1
- nn.ReLU() 활성화 함수
- Fully Connected Layer
  - 입력 특징 수: 16 * 28 * 28
  - 출력 노드 수: 10
- forward 함수를 정의
  - CNN의 출력 차원이 올바르게 맞춰지도록 view() 또는 reshape()을 활용

In [ ]:
import torch.nn as nn

class SimpleCNN(nn.Module):
  def __init__(self):
    super(SimpleCNN, self).__init__() # super 함수 선언 주의 !!

    self.conv2d = nn.Conv2d(1, 16, kernel_size=3, padding=1)
    self.relu = nn.ReLU()
    self.fc = nn.Linear((16*28*28), 10)

  def forward(self, x): # 함수 파라미터 (self, x) 주의!
    x = self.conv2d(x)  # x =  주의!
    x = self.relu(x)
    x = x.view(x.size(0), -1) # flatten (2d->1d) <== flatten 추가 !! 주의!!
    x = self.fc(x)
    return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
my_simple_model = SimpleCNN().to(device)

cuda


**손실 함수 및 옵티마이저 설정:**

- CrossEntropyLoss를 손실 함수 사용
- Adam 옵티마이저로 설정
- 학습률(lr): 0.001

In [ ]:
import torch.optim as optim

optimiser = optim.Adam(params=my_simple_model.parameters(), lr=0.001) # optimiser 세팅시, params 빠지지 않도록 주의!
loss_fn = nn.CrossEntropyLoss()

**모델 학습**
- 3 epochs 이상 학습하여 성능을 개선
- 학습이 완료된 후 "Training Complete!"를 출력

In [ ]:
# train
epochs = 50

for ep in range(1, epochs+1):
  running_loss = 0.0
  for imgs, labels in train_loader:
    imgs, labels = imgs.to(device), labels.to(device)
    optimiser.zero_grad()

    outputs = my_simple_model(imgs)
    loss = loss_fn(outputs, labels)
    loss.backward()

    optimiser.step

    running_loss += loss.item()

  if ep % 10 == 0:
    print(f"Epoch: {ep} - Loss: {running_loss/len(train_loader):.4f}")

print("Training Complete!")


Epoch: 10 - Loss: 2.3313
Epoch: 20 - Loss: 2.3313
Epoch: 30 - Loss: 2.3313
Epoch: 40 - Loss: 2.3313
Epoch: 50 - Loss: 2.3313
Training Complete!


**추가 도전 과제**
- 추가적인 CNN 계층(예: Conv2d 및 MaxPool2d)을 추가하여 모델을 개선

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms

# MNIST DataLoad
batch_size = 100
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class ImprovedCNN(nn.Module):
    def __init__(self):
        super(ImprovedCNN, self).__init__() # super 함수 선언 주의 !!

        # 첫 번째 합성곱 계층: 1채널 → 16채널, 커널 크기 3x3, 패딩 1
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)  # 28x28 → 14x14

        # 두 번째 합성곱 계층: 16채널 → 32채널
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)  # 14x14 → 7x7

        # Fully Connected Layer (FC)
        self.fc1 = nn.Linear(32 * 7 * 7, 128)  # FC 계층 (입력: 32*7*7)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)  # 출력 클래스 (0~9)

    def forward(self, x): # 함수 파라미터 (self, x) 주의!
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool2(x)

        x = x.view(x.size(0), -1) # flatten (2d->1d) <== flatten 추가 !! 주의!!
        x = self.fc1(x)
        x = self.relu3(x)
        x = self.fc2(x)
        return F.log_softmax(x) # fully-connected layer에 넣고 logsoftmax 적용
        # return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
my_improved_model = ImprovedCNN().to(device)

cuda


In [ ]:
import torch.optim as optim

optimiser = optim.Adam(params=my_improved_model.parameters(), lr=0.001) # optimiser 세팅시, params 빠지지 않도록 주의!
loss_fn = nn.CrossEntropyLoss().to(device)

In [ ]:
# train
epochs = 50

for ep in range(1, epochs+1):
  running_loss = 0.0
  for imgs, labels in train_loader:
    imgs, labels = imgs.to(device), labels.to(device)
    optimiser.zero_grad()

    outputs = my_improved_model(imgs)
    loss = loss_fn(outputs, labels)
    loss.backward()

    optimiser.step

    running_loss += loss.item()

  if ep % 10 == 0:
    print(f"Epoch: {ep} - Loss: {running_loss/len(train_loader):.4f}")

print("Training Complete!")


C:\Users\PC\AppData\Local\Temp\ipykernel_20964\2960065624.py:36: UserWarning: Implicit dimension choice for log_softmax has been deprecated. Change the call to include dim=X as an argument.
  return F.log_softmax(x) # fully-connected layer에 넣고 logsoftmax 적용


Epoch: 10 - Loss: 2.3029
Epoch: 20 - Loss: 2.3029
Epoch: 30 - Loss: 2.3029
Epoch: 40 - Loss: 2.3029
Epoch: 50 - Loss: 2.3029
Training Complete!


## **Final**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transfroms

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(777)
if device == 'cuda':
    torch.cuda.manual_seed_all(777)
print(device + " is available")

learning_rate = 0.001
batch_size = 100
num_classes = 10
epochs = 5

# MNIST 데이터셋 로드
train_set = torchvision.datasets.MNIST(
    root = './data/MNIST',
    train = True,
    download = True,
    transform = transfroms.Compose([
        transfroms.ToTensor() # 데이터를 0에서 255까지 있는 값을 0에서 1사이 값으로 변환
    ])
)
test_set = torchvision.datasets.MNIST(
    root = './data/MNIST',
    train = False,
    download = True,
    transform = transfroms.Compose([
        transfroms.ToTensor() # 데이터를 0에서 255까지 있는 값을 0에서 1사이 값으로 변환
    ])
)

# train_loader, test_loader 생성
train_loader = torch.utils.data.DataLoader(train_set, batch_size=batch_size)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=batch_size)

# input size를 알기 위해서
examples = enumerate(train_set)
batch_idx, (example_data, example_targets) = next(examples)
example_data.shape

class ConvNet(nn.Module):
  def __init__(self): # layer 정의
        super(ConvNet, self).__init__()

        # input size = 28x28
        self.conv1 = nn.Conv2d(1, 10, kernel_size=5) # input channel = 1, filter = 10, kernel size = 5, zero padding = 0, stribe = 1
        # ((W-K+2P)/S)+1 공식으로 인해 ((28-5+0)/1)+1=24 -> 24x24로 변환
        # maxpooling하면 12x12

        self.conv2 = nn.Conv2d(10, 20, kernel_size=5) # input channel = 1, filter = 10, kernel size = 5, zero padding = 0, stribe = 1
        # ((12-5+0)/1)+1=8 -> 8x8로 변환
        # maxpooling하면 4x4

        self.drop2D = nn.Dropout2d(p=0.25, inplace=False) # 랜덤하게 뉴런을 종료해서 학습을 방해해 학습이 학습용 데이터에 치우치는 현상을 막기 위해 사용
        self.mp = nn.MaxPool2d(2)  # 오버피팅을 방지하고, 연산에 들어가는 자원을 줄이기 위해 maxpolling
        self.fc1 = nn.Linear(320,100) # 4x4x20 vector로 flat한 것을 100개의 출력으로 변경
        self.fc2 = nn.Linear(100,10) # 100개의 출력을 10개의 출력으로 변경

  def forward(self, x):
        x = F.relu(self.mp(self.conv1(x))) # convolution layer 1번에 relu를 씌우고 maxpool, 결과값은 12x12x10
        x = F.relu(self.mp(self.conv2(x))) # convolution layer 2번에 relu를 씌우고 maxpool, 결과값은 4x4x20
        x = self.drop2D(x)
        x = x.view(x.size(0), -1) # flat
        x = self.fc1(x) # fc1 레이어에 삽입
        x = self.fc2(x) # fc2 레이어에 삽입
        return F.log_softmax(x) # fully-connected layer에 넣고 logsoftmax 적용

model = ConvNet().to(device) # CNN instance 생성
# Cost Function과 Optimizer 선택
criterion = nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)

for epoch in range(epochs): # epochs수만큼 반복
    avg_cost = 0

    for data, target in train_loader:
        data = data.to(device)
        target = target.to(device)
        optimizer.zero_grad() # 모든 model의 gradient 값을 0으로 설정
        hypothesis = model(data) # 모델을 forward pass해 결과값 저장
        cost = criterion(hypothesis, target) # output과 target의 loss 계산
        cost.backward() # backward 함수를 호출해 gradient 계산
        optimizer.step() # 모델의 학습 파라미터 갱신
        avg_cost += cost / len(train_loader) # loss 값을 변수에 누적하고 train_loader의 개수로 나눔 = 평균
    print('[Epoch: {:>4}] cost = {:>.9}'.format(epoch + 1, avg_cost))

# test
model.eval() # evaluate mode로 전환 dropout 이나 batch_normalization 해제
with torch.no_grad(): # grad 해제
    correct = 0
    total = 0

    for data, target in test_loader:
        data = data.to(device)
        target = target.to(device)
        out = model(data)
        preds = torch.max(out.data, 1)[1] # 출력이 분류 각각에 대한 값으로 나타나기 때문에, 가장 높은 값을 갖는 인덱스를 추출
        total += len(target) # 전체 클래스 개수
        correct += (preds==target).sum().item() # 예측값과 실제값이 같은지 비교

    print('Test Accuracy: ', 100.*correct/total, '%')


cuda is available


100.0%
100.0%
100.0%
100.0%
C:\Users\PC\AppData\Local\Temp\ipykernel_20964\2714754914.py:71: UserWarning: Implicit dimension choice for log_softmax has been deprecated. Change the call to include dim=X as an argument.
  return F.log_softmax(x) # fully-connected layer에 넣고 logsoftmax 적용


[Epoch:    1] cost = 0.316505849
[Epoch:    2] cost = 0.117470078
[Epoch:    3] cost = 0.089020282
[Epoch:    4] cost = 0.0758361071
[Epoch:    5] cost = 0.0648855194
Test Accuracy:  98.58 %
